# Sesión 03 — Métricas de Clasificación y AUROC
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo I · Fundamentos del Aprendizaje Estadístico**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Derivar exactitud, precisión, sensibilidad, especificidad, F1 y exactitud balanceada a partir de la matriz de confusión.
2. Construir la curva ROC paso a paso y calcular el AUROC.
3. Seleccionar el umbral óptimo según el criterio de Youden y según costos asimétricos.
4. Construir la curva precisión-exhaustividad (PR) y explicar cuándo preferir AUPRC sobre AUROC.
5. Calcular intervalos de confianza bootstrap para el AUROC y aplicar la prueba de DeLong para comparar dos clasificadores.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010 |
| ★★★ | Saito, T. & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432 |
| ★★☆ | DeLong, E.R., DeLong, D.M. & Clarke-Pearson, D.L. (1988). Comparing the areas under two or more correlated receiver operating characteristic curves. *Biometrics*, 44(3), 837–845. |
| ★★☆ | Hanley, J.A. & McNeil, B.J. (1982). The meaning and use of the area under a receiver operating characteristic (ROC) curve. *Radiology*, 143(1), 29–36. |
| ★☆☆ | Davis, J. & Goadrich, M. (2006). The relationship between precision-recall and ROC curves. *ICML 2006*. |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Configuración completa.')

## Parte 1 — La matriz de confusión y las métricas derivadas

Toda métrica de clasificación binaria se deriva de cuatro conteos fundamentales:

| | Predicho positivo | Predicho negativo |
|---|---|---|
| **Real positivo** | VP (verdadero positivo) | FN (falso negativo) |
| **Real negativo** | FP (falso positivo) | VN (verdadero negativo) |

Las métricas principales:

| Métrica | Fórmula | También llamada |
|---|---|---|
| Sensibilidad | VP / (VP+FN) | Recall, TPR, exhaustividad |
| Especificidad | VN / (VN+FP) | TNR, 1−FPR |
| Precisión | VP / (VP+FP) | VPP |
| Exactitud | (VP+VN) / N | Accuracy |
| F1 | 2·Prec·Rec / (Prec+Rec) | Media armónica de precisión y recall |
| Exactitud balanceada | (Sens+Espec) / 2 | Balanced accuracy |

In [ ]:
# ── Dataset: detección de fibrilación auricular (FA) con ECG de una derivación ──
# Simulado con parámetros basados en:
# Attia, Z.I. et al. (2019). Screening for cardiac contractile dysfunction using
# an artificial intelligence–enabled electrocardiogram. Nature Medicine, 25, 70–74.
# https://doi.org/10.1038/s41591-018-0240-2

N_pos = 180   # pacientes con FA confirmada
N_neg = 820   # pacientes en ritmo sinusal  (desbalance 1:4.5 — realista)
N     = N_pos + N_neg

# Scores del clasificador (probabilidad de FA)
scores_pos = rng.beta(7, 2, N_pos)   # FA: scores altos
scores_neg = rng.beta(2, 7, N_neg)   # No FA: scores bajos

scores = np.concatenate([scores_pos, scores_neg])
labels = np.concatenate([np.ones(N_pos), np.zeros(N_neg)])

# Función para calcular todas las métricas dado un umbral
def metricas_umbral(scores, labels, umbral):
    pred = (scores >= umbral).astype(int)
    VP = ((pred == 1) & (labels == 1)).sum()
    FP = ((pred == 1) & (labels == 0)).sum()
    FN = ((pred == 0) & (labels == 1)).sum()
    VN = ((pred == 0) & (labels == 0)).sum()
    sens  = VP / (VP + FN) if (VP + FN) > 0 else 0
    espec = VN / (VN + FP) if (VN + FP) > 0 else 0
    prec  = VP / (VP + FP) if (VP + FP) > 0 else 0
    exact = (VP + VN) / len(labels)
    f1    = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0
    bal   = (sens + espec) / 2
    return dict(VP=VP, FP=FP, FN=FN, VN=VN,
                Sensibilidad=sens, Especificidad=espec,
                Precisión=prec, Exactitud=exact, F1=f1,
                ExactitudBalanceada=bal)

# Evaluar con umbral = 0.5
m = metricas_umbral(scores, labels, umbral=0.5)

print('Métricas con umbral = 0.50')
print('─' * 45)
print(f'  Matriz de confusión:')
print(f'    VP={m["VP"]:4d}  FP={m["FP"]:4d}')
print(f'    FN={m["FN"]:4d}  VN={m["VN"]:4d}')
print()
for k in ['Sensibilidad','Especificidad','Precisión','Exactitud','F1','ExactitudBalanceada']:
    print(f'  {k:<22} = {m[k]:.3f}')

# Visualizar la matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
matriz = np.array([[m['VP'], m['FN']], [m['FP'], m['VN']]])
im = ax.imshow(matriz, cmap='Blues')
etiquetas = [['VP\n(verdadero+)', 'FN\n(falso-)'],
              ['FP\n(falso+)',     'VN\n(verdadero-)']]
for i in range(2):
    for j in range(2):
        color = 'white' if matriz[i,j] > matriz.max()*0.6 else '#1B3A6B'
        ax.text(j, i, f'{etiquetas[i][j]}\n{matriz[i,j]}',
                ha='center', va='center', fontsize=12,
                color=color, fontweight='bold')
ax.set(xticks=[0,1], yticks=[0,1],
       xticklabels=['Predicho FA', 'Predicho No FA'],
       yticklabels=['Real FA', 'Real No FA'],
       title='Matriz de confusión — umbral=0.50')
plt.tight_layout()
plt.show()

## Parte 2 — La curva ROC y el AUROC

La curva ROC traza la **tasa de verdaderos positivos (TPR = sensibilidad)** contra la
**tasa de falsos positivos (FPR = 1 − especificidad)** para todos los umbrales posibles.

El **AUROC** (área bajo la curva ROC) tiene una interpretación probabilística elegante:

$$\text{AUROC} = P(\text{score}_{+} > \text{score}_{-})$$

Es la probabilidad de que un ejemplo positivo elegido al azar tenga un score más alto
que un ejemplo negativo elegido al azar.  
- AUROC = 1.0 → clasificador perfecto  
- AUROC = 0.5 → clasificador aleatorio  
- AUROC = 0.0 → clasificador perfectamente invertido

In [ ]:
# ── Construcción manual de la curva ROC ───────────────────────────────────────
def curva_roc(scores, labels):
    """Construye la curva ROC ordenando por score descendente."""
    orden  = np.argsort(scores)[::-1]
    s_ord  = scores[orden]
    l_ord  = labels[orden]

    n_pos = labels.sum()
    n_neg = len(labels) - n_pos

    tprs = [0.0]
    fprs = [0.0]
    umbrales = []

    tp, fp = 0, 0
    prev_score = np.inf

    for i, (s, l) in enumerate(zip(s_ord, l_ord)):
        if s != prev_score and i > 0:
            tprs.append(tp / n_pos)
            fprs.append(fp / n_neg)
            umbrales.append(prev_score)
        if l == 1:
            tp += 1
        else:
            fp += 1
        prev_score = s

    tprs.append(tp / n_pos)
    fprs.append(fp / n_neg)
    umbrales.append(prev_score)
    return np.array(fprs), np.array(tprs), np.array(umbrales)

def auroc_trapecio(fprs, tprs):
    """Calcula el AUROC por la regla del trapecio."""
    return np.trapz(tprs, fprs)


fprs, tprs, umbs = curva_roc(scores, labels)
auroc = auroc_trapecio(fprs, tprs)

# Umbral óptimo de Youden: maximiza Sensibilidad + Especificidad − 1
youden   = tprs - fprs
idx_you  = np.argmax(youden)
umb_you  = umbs[idx_you]
tpr_you  = tprs[idx_you]
fpr_you  = fprs[idx_you]

print(f'AUROC = {auroc:.4f}')
print(f'Umbral óptimo de Youden = {umb_you:.3f}')
print(f'  TPR (sensibilidad) = {tpr_you:.3f}')
print(f'  FPR (1-espec.)     = {fpr_you:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Curva ROC
axes[0].plot(fprs, tprs, 'steelblue', lw=2.5,
              label=f'Clasificador FA (AUROC={auroc:.3f})')
axes[0].plot([0,1],[0,1],'gray',ls='--',lw=1.2, label='Aleatorio (AUROC=0.5)')
axes[0].scatter([fpr_you],[tpr_you], s=120, color='tomato', zorder=5,
                 label=f'Youden: umb={umb_you:.2f}\nSens={tpr_you:.2f}, Espec={1-fpr_you:.2f}')
axes[0].fill_between(fprs, tprs, alpha=0.1, color='steelblue')
axes[0].set(xlabel='FPR (1 − Especificidad)', ylabel='TPR (Sensibilidad)',
            title='Curva ROC — Detección de FA',
            xlim=[-0.02,1.02], ylim=[-0.02,1.02])
axes[0].legend(fontsize=8)
axes[0].set_aspect('equal')

# Distribución de scores por clase
axes[1].hist(scores_neg, bins=30, alpha=0.5, color='steelblue',
              density=True, label='No FA (negativo)')
axes[1].hist(scores_pos, bins=30, alpha=0.5, color='tomato',
              density=True, label='FA (positivo)')
axes[1].axvline(umb_you, color='k', ls='--', lw=2,
                 label=f'Umbral Youden={umb_you:.2f}')
axes[1].set(xlabel='Score del clasificador', ylabel='Densidad',
            title='Distribución de scores por clase\n'
                  'La superposición determina el AUROC')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Parte 3 — Curva precisión-exhaustividad (PR) y AUPRC

La curva PR traza **precisión** (VP/(VP+FP)) contra **exhaustividad** (recall = sensibilidad)
para todos los umbrales.

**¿Cuándo preferir AUPRC sobre AUROC?**

> Cuando la clase positiva es **muy minoritaria** (alta desbalanza), el AUROC puede ser
> optimista porque los VN abundantes inflan la especificidad.
> El AUPRC es más sensible al rendimiento en la clase minoritaria.

Regla práctica (Saito & Rehmsmeier, 2015):
- Desbalance < 1:10 → AUROC es fiable
- Desbalance ≥ 1:10 → reportar ambos, priorizar AUPRC

In [ ]:
# ── Curva PR y comparación AUROC vs AUPRC ──────────────────────────────────────
def curva_pr(scores, labels):
    """Construye la curva precisión-exhaustividad."""
    orden = np.argsort(scores)[::-1]
    l_ord = labels[orden]
    n_pos = labels.sum()

    tp, fp = 0, 0
    precs, recs = [], []

    for l in l_ord:
        if l == 1:
            tp += 1
        else:
            fp += 1
        precs.append(tp / (tp + fp))
        recs.append(tp / n_pos)

    # Añadir punto inicial (recall=0, precisión=1)
    return np.array([0] + recs), np.array([1] + precs)


# Comparar tres escenarios de desbalance
desbalances = [
    (180, 820,  '1:4.5 (FA en clínica general)',    'steelblue'),
    (50,  950,  '1:19  (FA en población joven)',     'tomato'),
    (10,  990,  '1:99  (FA en <40 años)',            'seagreen'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for n_p, n_n, etiqueta, color in desbalances:
    s_p = rng.beta(7, 2, n_p)
    s_n = rng.beta(2, 7, n_n)
    s   = np.concatenate([s_p, s_n])
    l   = np.concatenate([np.ones(n_p), np.zeros(n_n)])

    # ROC
    fpr, tpr, _ = curva_roc(s, l)
    auc_roc = auroc_trapecio(fpr, tpr)
    axes[0].plot(fpr, tpr, lw=2, color=color,
                  label=f'{etiqueta}\nAUROC={auc_roc:.3f}')

    # PR
    rec, prec = curva_pr(s, l)
    auc_pr = np.trapz(prec, rec)
    prev   = n_p / (n_p + n_n)
    axes[1].plot(rec, prec, lw=2, color=color,
                  label=f'{etiqueta}\nAUPRC={auc_pr:.3f}  (prev={prev:.1%})')
    # Línea base (clasificador aleatorio) en curva PR = prevalencia
    axes[1].axhline(prev, color=color, ls=':', lw=1, alpha=0.5)

axes[0].plot([0,1],[0,1],'gray',ls='--',lw=1)
axes[0].set(xlabel='FPR', ylabel='TPR',
            title='Curva ROC — efecto del desbalance\n'
                  'AUROC permanece relativamente estable',
            xlim=[-0.02,1.02], ylim=[-0.02,1.02])
axes[0].legend(fontsize=7.5)
axes[0].set_aspect('equal')

axes[1].set(xlabel='Exhaustividad (Recall)', ylabel='Precisión',
            title='Curva PR — efecto del desbalance\n'
                  'AUPRC colapsa con alta desbalanza',
            xlim=[-0.02,1.02], ylim=[-0.02,1.02])
axes[1].legend(fontsize=7.5)

plt.suptitle('AUROC vs AUPRC bajo distintos niveles de desbalance de clases\n'
              'Los puntos discontinuos en la curva PR = clasificador aleatorio (prevalencia)',
              fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print('Observación:')
print('  El AUROC cambia poco con el desbalance porque los VN abundantes')
print('  mantienen la especificidad alta.')
print('  El AUPRC colapsa porque la precisión se degrada cuando hay pocos positivos.')

## Parte 4 — Selección de umbral con costos asimétricos

En aplicaciones clínicas los costos de FP y FN son raramente iguales:

- **Tamizaje:** costo(FN) >> costo(FP) → priorizar sensibilidad
- **Confirmación diagnóstica:** costo(FP) >> costo(FN) → priorizar especificidad

El umbral óptimo minimiza la función de costo esperada:

$$\text{Costo}(u) = C_{FN} \cdot \text{FN}(u) + C_{FP} \cdot \text{FP}(u)$$

In [ ]:
# ── Selección de umbral con costos asimétricos ─────────────────────────────────
umbrales_grid = np.linspace(0.01, 0.99, 200)

# Tres escenarios de costos
escenarios_costo = [
    (1, 1,   'Costos iguales C_FN=1, C_FP=1',         'steelblue'),
    (5, 1,   'Tamizaje C_FN=5, C_FP=1 (FN más caro)', 'tomato'),
    (1, 5,   'Confirmación C_FN=1, C_FP=5 (FP más caro)', 'seagreen'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for c_fn, c_fp, etiqueta, color in escenarios_costo:
    costos = []
    sens_v, spec_v = [], []
    for u in umbrales_grid:
        m_u = metricas_umbral(scores, labels, u)
        costos.append(c_fn * m_u['FN'] + c_fp * m_u['FP'])
        sens_v.append(m_u['Sensibilidad'])
        spec_v.append(m_u['Especificidad'])

    costos  = np.array(costos)
    idx_opt = np.argmin(costos)
    u_opt   = umbrales_grid[idx_opt]

    axes[0].plot(umbrales_grid, costos / costos.max(), lw=2,
                  color=color, label=f'{etiqueta}\nÓptimo: u={u_opt:.2f}')
    axes[0].axvline(u_opt, color=color, ls='--', lw=1, alpha=0.6)

    # Sens y Espec vs umbral
    if c_fn == 5:
        axes[1].plot(umbrales_grid, sens_v, 'tomato', lw=2, label='Sensibilidad')
        axes[1].plot(umbrales_grid, spec_v, 'steelblue', lw=2, label='Especificidad')
        axes[1].axvline(u_opt, color='tomato', ls='--', lw=1.5,
                         label=f'Óptimo tamizaje: {u_opt:.2f}')
        # Youden
        youden_vals = np.array(sens_v) + np.array(spec_v) - 1
        u_youden    = umbrales_grid[np.argmax(youden_vals)]
        axes[1].axvline(u_youden, color='k', ls=':', lw=1.5,
                         label=f'Youden: {u_youden:.2f}')

axes[0].set(xlabel='Umbral', ylabel='Costo normalizado',
            title='Costo total vs umbral — tres escenarios clínicos\n'
                  'El óptimo se desplaza según la relación C_FN/C_FP')
axes[0].legend(fontsize=7.5)

axes[1].set(xlabel='Umbral', ylabel='Tasa',
            title='Sensibilidad y especificidad vs umbral\n'
                  'Trade-off fundamental de cualquier clasificador binario')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Mensaje clave:')
print('  No existe un umbral universalmente óptimo.')
print('  El umbral correcto depende del contexto clínico y de los costos relativos.')
print('  Youden = costos iguales. En tamizaje (FN caro), umbral debe ser más bajo.')

## Parte 5 — Intervalos de confianza bootstrap para el AUROC

Un AUROC puntual sin intervalo de confianza es un resultado incompleto.
El bootstrap no paramétrico es el método más robusto para obtener el IC del AUROC
sin asumir ninguna distribución de los scores.

In [ ]:
# ── Bootstrap para el AUROC ───────────────────────────────────────────────────
def auroc_bootstrap(scores, labels, n_boot=2000, ci=0.95, rng_=None):
    """IC bootstrap para el AUROC mediante remuestreo estratificado."""
    rng_ = rng_ or np.random.default_rng()
    n    = len(labels)
    idx_pos = np.where(labels == 1)[0]
    idx_neg = np.where(labels == 0)[0]

    aurocs_boot = []
    for _ in range(n_boot):
        # Remuestreo estratificado (mantiene la proporción de clases)
        idx_p = rng_.choice(idx_pos, len(idx_pos), replace=True)
        idx_n = rng_.choice(idx_neg, len(idx_neg), replace=True)
        idx_b = np.concatenate([idx_p, idx_n])

        s_b = scores[idx_b]
        l_b = labels[idx_b]
        fpr_b, tpr_b, _ = curva_roc(s_b, l_b)
        aurocs_boot.append(auroc_trapecio(fpr_b, tpr_b))

    alpha = 1 - ci
    lo    = np.percentile(aurocs_boot, 100 * alpha / 2)
    hi    = np.percentile(aurocs_boot, 100 * (1 - alpha / 2))
    return np.array(aurocs_boot), lo, hi


boot_aurocs, lo95, hi95 = auroc_bootstrap(scores, labels, n_boot=2000, rng_=rng)

print(f'AUROC = {auroc:.4f}  IC95% [{lo95:.4f}, {hi95:.4f}]')
print(f'Anchura del IC: {hi95-lo95:.4f}')

# Efecto del tamaño muestral sobre el IC
ns_effect = [50, 100, 200, 500, 1000]
ic_widths  = []
for n_sub in ns_effect:
    # Submuestrear manteniendo la proporción de clases
    n_p_sub = max(5, round(n_sub * N_pos / N))
    n_n_sub = n_sub - n_p_sub
    idx_p   = rng.choice(np.where(labels==1)[0], n_p_sub, replace=False)
    idx_n   = rng.choice(np.where(labels==0)[0], n_n_sub, replace=False)
    idx_sub = np.concatenate([idx_p, idx_n])
    _, lo_s, hi_s = auroc_bootstrap(scores[idx_sub], labels[idx_sub],
                                     n_boot=500, rng_=rng)
    ic_widths.append(hi_s - lo_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribución bootstrap
axes[0].hist(boot_aurocs, bins=40, density=True,
              color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(auroc, color='navy',  lw=2.5, label=f'AUROC={auroc:.4f}')
axes[0].axvline(lo95,  color='tomato',ls='--',lw=2,  label=f'IC95% [{lo95:.3f}, {hi95:.3f}]')
axes[0].axvline(hi95,  color='tomato',ls='--',lw=2)
axes[0].fill_betweenx(
    [0, axes[0].get_ylim()[1] if axes[0].get_ylim()[1] > 0 else 30],
    lo95, hi95, alpha=0.1, color='tomato'
)
axes[0].set(xlabel='AUROC (bootstrap)', ylabel='Densidad',
            title='Distribución bootstrap del AUROC\n'
                  f'N={N}, 2000 remuestras estratificadas')
axes[0].legend(fontsize=9)

# IC vs tamaño muestral
axes[1].plot(ns_effect, ic_widths, 'tomato', lw=2.5, marker='o', ms=8)
axes[1].set(xlabel='Tamaño muestral N', ylabel='Anchura del IC95%',
            xscale='log',
            title='Anchura del IC95% del AUROC vs N\n'
                  'Decrece como ≈ 1/√N — necesitas 4× más datos para reducir a la mitad')
for n_s, w in zip(ns_effect, ic_widths):
    axes[1].annotate(f'N={n_s}\n{w:.3f}', (n_s, w),
                      textcoords='offset points', xytext=(5, 5), fontsize=8)

plt.tight_layout()
plt.show()

## Parte 6 — Prueba de DeLong: comparar dos clasificadores

La prueba de DeLong (1988) compara los AUROCs de dos clasificadores sobre el mismo
conjunto de datos teniendo en cuenta la **correlación** entre las curvas (los mismos
pacientes fueron evaluados por ambos). Es más potente que un bootstrap independiente.

In [ ]:
# ── Prueba de DeLong simplificada ──────────────────────────────────────────────
# Comparamos dos algoritmos de detección de FA:
# A: modelo basado en solo la FC y el intervalo RR
# B: modelo con más características morfológicas del ECG

# Simular scores de dos clasificadores correlacionados
# (mismos pacientes, distintos modelos)
r = 0.65   # correlación entre los dos modelos

# Generar scores correlacionados mediante transformación de Cholesky
def scores_correlacionados(scores_base, r, ruido, rng_):
    """Genera un segundo clasificador correlacionado con el primero."""
    s2 = r * scores_base + np.sqrt(1 - r**2) * rng_.normal(0, ruido, len(scores_base))
    return np.clip(s2, 0, 1)

scores_A = scores   # clasificador original
scores_B = scores_correlacionados(scores, r=0.65, ruido=0.12, rng_=rng)
scores_B = np.clip(scores_B + 0.04, 0, 1)   # B es ligeramente mejor

fpr_A, tpr_A, _ = curva_roc(scores_A, labels)
fpr_B, tpr_B, _ = curva_roc(scores_B, labels)
auroc_A = auroc_trapecio(fpr_A, tpr_A)
auroc_B = auroc_trapecio(fpr_B, tpr_B)

# Estadístico de DeLong (aproximación no paramétrica)
def delong_test(scores_1, scores_2, labels):
    """
    Prueba de DeLong para comparar dos AUROCs correlacionados.
    Retorna (z, p_valor, auroc_1, auroc_2)
    """
    idx_p = np.where(labels == 1)[0]
    idx_n = np.where(labels == 0)[0]
    n_p, n_n = len(idx_p), len(idx_n)

    def componentes_varianza(s, idx_p, idx_n):
        # Componente de varianza estructural (Hanley & McNeil 1982)
        V10 = np.array([np.mean(s[idx_p] > s[j]) for j in idx_n])
        V01 = np.array([np.mean(s[i] > s[idx_n]) for i in idx_p])
        auc = np.mean(V10)
        s10 = np.var(V10, ddof=1)
        s01 = np.var(V01, ddof=1)
        return auc, s10, s01, V10, V01

    auc1, s10_1, s01_1, V10_1, V01_1 = componentes_varianza(scores_1, idx_p, idx_n)
    auc2, s10_2, s01_2, V10_2, V01_2 = componentes_varianza(scores_2, idx_p, idx_n)

    # Covarianza entre los dos AUROCs
    cov10 = np.cov(V10_1, V10_2, ddof=1)[0,1]
    cov01 = np.cov(V01_1, V01_2, ddof=1)[0,1]

    # Varianza de la diferencia
    var_diff = (s10_1/n_n + s01_1/n_p) + (s10_2/n_n + s01_2/n_p) \
               - 2*(cov10/n_n + cov01/n_p)
    var_diff = max(var_diff, 1e-10)   # evitar división por cero

    z = (auc1 - auc2) / np.sqrt(var_diff)
    p = 2 * stats.norm.sf(abs(z))
    return z, p, auc1, auc2


z, p_val, auc_A, auc_B = delong_test(scores_A, scores_B, labels)

print('Comparación de clasificadores — Prueba de DeLong')
print('─' * 50)
print(f'  AUROC modelo A (FC+RR básico)      = {auc_A:.4f}')
print(f'  AUROC modelo B (ECG morfológico)   = {auc_B:.4f}')
print(f'  Diferencia AUROC                   = {auc_B-auc_A:.4f}')
print(f'  Estadístico z                      = {z:.3f}')
print(f'  p-valor (bilateral)                = {p_val:.4f}')
print()
if p_val < 0.05:
    print('  → Diferencia estadísticamente significativa (p<0.05)')
    print('    El modelo B es significativamente mejor que A.')
else:
    print('  → Diferencia NO estadísticamente significativa (p≥0.05)')
    print('    No hay evidencia suficiente para afirmar que B sea mejor.')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_A, tpr_A, 'steelblue', lw=2.5,
         label=f'Modelo A — FC+RR  (AUROC={auc_A:.3f})')
ax.plot(fpr_B, tpr_B, 'tomato',    lw=2.5,
         label=f'Modelo B — ECG morfológico  (AUROC={auc_B:.3f})')
ax.plot([0,1],[0,1],'gray',ls='--',lw=1)
ax.fill_between(fpr_B, tpr_A, tpr_B, alpha=0.12, color='tomato',
                 label=f'Diferencia  (Δ={auc_B-auc_A:.3f}, p={p_val:.3f})')
ax.set(xlabel='FPR (1−Especificidad)', ylabel='TPR (Sensibilidad)',
       title=f'Comparación ROC — Prueba de DeLong\n'
              f'N={N}  |  Correlación entre modelos r≈{r}',
       xlim=[-0.02,1.02], ylim=[-0.02,1.02])
ax.legend(fontsize=9)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Exactitud engañosa.** Con los datos de detección de FA de esta sesión, evalúa un
   clasificador trivial que siempre predice "No FA".  
   (a) Calcula su exactitud, sensibilidad, especificidad y F1.  
   (b) ¿Por qué la exactitud es "alta" aunque el clasificador sea inútil?  
   (c) ¿Qué métrica detecta mejor este problema?

2. **Curva ROC con distintas poblaciones.** Simula el mismo clasificador de FA aplicado
   en tres contextos clínicos: (i) consulta general (prev=2%), (ii) unidad de arritmias
   (prev=30%), (iii) UCI cardiológica (prev=60%). Construye las curvas ROC y PR para cada
   contexto. ¿Cuál cambia y cuál permanece constante? Explica por qué.

3. **IC de DeLong vs bootstrap.** Compara el IC95% del AUROC obtenido por bootstrap
   (Parte 5) con el IC derivado de la prueba de DeLong (suponiendo normalidad asintótica
   del estadístico z). ¿Son similares? ¿Cuál es más conservador?

4. **Análisis costo-beneficio.** Para el clasificador de FA con C_FN=5 y C_FP=1,
   calcula el umbral óptimo para prevalencias de 2%, 10% y 30%. ¿Cómo cambia el
   umbral con la prevalencia? ¿Tiene sentido clínico ese cambio?

5. *(Desafío)* **AUROC del conjunto PTB-XL.** Descarga el conjunto PTB-XL
   (physionet.org/content/ptb-xl/) y entrena un clasificador simple (regresión logística
   sobre características espectrales) para distinguir FA vs ritmo sinusal. Reporta
   AUROC con IC95% bootstrap y compara con la línea base aleatoria usando la prueba de
   DeLong.

## 📚 Conjuntos de datos utilizados / referenciados

| Conjunto de datos | Fuente | Notas |
|---|---|
| Detección de FA (scores simulados) | Attia, Z.I. et al. (2019). *Nature Medicine*, 25, 70–74. https://doi.org/10.1038/s41591-018-0240-2 | Parámetros de desbalance calibrados con este estudio |
| PhysioNet PTB-XL | Wagner, P. et al. (2020). *Nature Scientific Data*, 7, 154. https://doi.org/10.1038/s41597-020-0495-6 | ECG de 12 derivaciones, 71 etiquetas diagnósticas |
| MIT-BIH Arrhythmia | Moody, G.B. & Mark, R.G. (2001). *IEEE Engineering in Medicine and Biology*, 20(3), 45–50. https://doi.org/10.13026/C2F305 | Referencia estándar para clasificación de arritmias |